In [ ]:
'''
Script to generate the the tables of the paper (TSV format)

This script creates the all the tables of the paper. It saves them as TSV
files.

Date: May-2026 (created)

'''

import os
import pandas as pd
import numpy as np

print(os.getcwd())
datadir = os.path.abspath(os.path.join(os.path.dirname( os.getcwd() ), '.', 'data'))
print(f'Main directory: {datadir}')

analyses_dict = {
    'basic_main': {'stat_folder': 'lme_maxT_finley_2betas_10000rand', 'trans': True},
    'onecov_emptyroom': {'stat_folder': 'lme_1cov_emptyroomtotalminusaperiodicrest_maxT_finley_2betas_ecg04eog08_10000rand', 'trans': True},
    'onecov_ecg': {'stat_folder': 'lme_1cov_ecglikemegtotalrelpow2-40Hz_maxT_finley_2betas_ecg04eog08_10000rand', 'trans': True},
    'sixcov': {'stat_folder': 'lme_allcov_maxT_finley_2betas_ecg04eog08_10000rand', 'trans': True},
    'control_ignorepeaks': {'stat_folder': 'lme_maxT_finley_2betas_nonoisepeaks_ecg04eog08_10000rand', 'trans': True},
    'control_nointerpol': {'stat_folder': 'lme_maxT_finley_2betas_nointerp_ecg04eog08_10000rand', 'trans': True},
    'control_cardiacMEGonly': {'stat_folder': 'lme_maxT_finley_2betas_allbutecg04_10000rand', 'trans': True},
    'control_withcardiac': {'stat_folder': 'lme_maxT_finley_2betas_eog08_10000rand', 'trans': True},
    'control_ECGchannel': {'stat_folder': 'lme_maxT_finley_2betas_ECGlikemeg_10000rand', 'trans': False},
    'control_emptyroom': {'stat_folder': 'lme_maxT_finley_2betas_filt_emptyroom_10000rand', 'trans': False},
}

megtype = 'grad'

# Loop over spectral parameters of interest and create dataframes with the main statistics

parameters = ['exponent']
bands = ['theta', 'alpha', 'beta', 'gamma', 'low_alpha', 'high_alpha', 'low_beta', 'high_beta']

for band in bands:
    for measure in ['band_power', 'peak_freq']:
        parameters.append(f'{band}_{measure}')


df_source = pd.read_csv(os.path.join(datadir, f'tables_source_data_grad.tsv'), sep='\t')

def process_beta(beta):
    beta_str = f'{beta:.2g}'.replace('e', ' x 10')  # replace 'e' with ' x 10'
    for pow in range(4, 10):
        beta_str = beta_str.replace(f' x 10-0{pow}', f' x 10-{pow}')
    return beta_str

def process_pval(pval):
    if pval == 0:
        return '< 10-4'
    pval_str = f'{pval:.2g}'.replace('e', ' x 10')  # replace 'e' with ' x 10'
    for pow in range(4, 10):
        pval_str = pval_str.replace(f' x 10-0{pow}', f' x 10-{pow}')
    return pval_str

def create_simple_table(df_source, parameter_label, parameter):

    units = 'Hz' if 'peak_freq' in parameter_label else 'a.u.'
    
    # Create a new pandas DataFrame to hold the table data (4 columns)
    column_names = [f'{parameter} Model / Age effect', f'Parameter Estimate, {units} p.a. (SD)', 'T-statistic (df)', 'P-value uncorrected (corrected)']   

    models = ['Basic', 'Empty-room', 'Cardiac', '6-covariate'] # labels in new table (paper)
    models_labels = ['basic_main', 'onecov_emptyroom', 'onecov_ecg', 'sixcov'] # labels in source data file

    effects = ['A0', 'dA', 'A0:dA'] # as they will appear in the new table (paper)
    effects_labels = ['Age0', 'deltaAge', 'Age0:deltaAge'] # as they appear in the source data file 
    
    # Table data
    df_list = []
    for m, model in enumerate(models):
        print(f'Processing {model} model...')

        df_tmp_list = [pd.DataFrame.from_dict({0: [model +  ' Model', '', '', '']}, orient='index', columns=column_names)]

        for e, effect in enumerate(effects):

            df_current = df_source.loc[(df_source['Parameter'] == parameter_label) & (df_source['Effect'] == effects_labels[e])]

            tmp_dict = {
                e+1: [effect, f"{process_beta(df_current [f'{models_labels[m]}_Estimate'].values[0])} ({process_beta(df_current [f'{models_labels[m]}_SE'].values[0])})",
                         f"{round(df_current [f'{models_labels[m]}_T-stat'].values[0], 2)} ({round(df_current [f'{models_labels[m]}_DF'].values[0], 1)})",
                         f"{process_pval(df_current [f'{models_labels[m]}_P-val'].values[0])} ({process_pval(df_current [f'{models_labels[m]}_P-corrected'].values[0])})"]
            }

            tmp_df = pd.DataFrame.from_dict(tmp_dict, orient='index', columns=column_names)
            df_tmp_list.append(tmp_df)

        df_tmp = pd.concat(df_tmp_list)
        df_list.append(df_tmp)
                        
    df_table = pd.concat(df_list, ignore_index=True)

    return df_table



# --- Table 1 ---

df_table = create_simple_table(df_source, 'exponent', 'Aperiodic Exponent')
df_table.to_csv(f'Table01_exponent.tsv', sep='\t', index=False)


/home/mc06/Documents/CamCAN/code/camcan_meglongrest_specparamlme_2026/results
Main directory: /home/mc06/Documents/CamCAN/code/camcan_meglongrest_specparamlme_2026/data
Processing Basic model...
Processing Empty-room model...
Processing Cardiac model...
Processing 6-covariate model...


In [ ]:
parameter = 'theta_band_power'
df_table = create_simple_table(df_source, parameter, 'Theta Power')
df_table.to_csv(f'Table02_{parameter}.tsv', sep='\t', index=False)

Processing Basic model...
Processing Empty-room model...
Processing Cardiac model...
Processing 6-covariate model...


In [ ]:
parameter = 'alpha_band_power'
df_table_left = create_simple_table(df_source, parameter, 'Alpha Power')
df_table_right = create_simple_table(df_source, 'alpha_peak_freq', 'Alpha Frequency')

df_table = pd.concat([df_table_left, df_table_right], axis=1)
df_table.to_csv(f'Table03_alpha.tsv', sep='\t', index=False)

Processing Basic model...
Processing Empty-room model...
Processing Cardiac model...
Processing 6-covariate model...
Processing Basic model...
Processing Empty-room model...
Processing Cardiac model...
Processing 6-covariate model...


In [ ]:
parameter = 'beta_band_power'
df_table_left = create_simple_table(df_source, parameter, 'Beta Power')
df_table_right = create_simple_table(df_source, 'beta_peak_freq', 'Beta Frequency')

df_table = pd.concat([df_table_left, df_table_right], axis=1)
df_table.to_csv(f'Table04_beta.tsv', sep='\t', index=False)

Processing Basic model...
Processing Empty-room model...
Processing Cardiac model...
Processing 6-covariate model...
Processing Basic model...
Processing Empty-room model...
Processing Cardiac model...
Processing 6-covariate model...


In [ ]:
parameter = 'gamma_band_power'
df_table_left = create_simple_table(df_source, parameter, 'Gamma Power')
df_table_right = create_simple_table(df_source, 'gamma_peak_freq', 'Gamma Frequency')

df_table = pd.concat([df_table_left, df_table_right], axis=1)
df_table.to_csv(f'Table05_gamma.tsv', sep='\t', index=False)

Processing Basic model...
Processing Empty-room model...
Processing Cardiac model...
Processing 6-covariate model...
Processing Basic model...
Processing Empty-room model...
Processing Cardiac model...
Processing 6-covariate model...


In [ ]:
parameter = 'low_alpha_band_power'
df_table_left = create_simple_table(df_source, parameter, 'Low-Alpha Power')
df_table_right = create_simple_table(df_source, 'low_alpha_peak_freq', 'Low-Alpha Frequency')

df_table = pd.concat([df_table_left, df_table_right], axis=1)
df_table.to_csv(f'Supp_Table01_low_alpha.tsv', sep='\t', index=False)

Processing Basic model...
Processing Empty-room model...
Processing Cardiac model...
Processing 6-covariate model...
Processing Basic model...
Processing Empty-room model...
Processing Cardiac model...
Processing 6-covariate model...


In [ ]:
parameter = 'high_alpha_band_power'
df_table_left = create_simple_table(df_source, parameter, 'High-Alpha Power')
df_table_right = create_simple_table(df_source, 'high_alpha_peak_freq', 'High-Alpha Frequency')

df_table = pd.concat([df_table_left, df_table_right], axis=1)
df_table.to_csv(f'Supp_Table02_high_alpha.tsv', sep='\t', index=False)

Processing Basic model...
Processing Empty-room model...
Processing Cardiac model...
Processing 6-covariate model...
Processing Basic model...
Processing Empty-room model...
Processing Cardiac model...
Processing 6-covariate model...


In [ ]:
parameter = 'low_beta_band_power'
df_table_left = create_simple_table(df_source, parameter, 'Low-Beta Power')
df_table_right = create_simple_table(df_source, 'low_beta_peak_freq', 'Low-Beta Frequency')

df_table = pd.concat([df_table_left, df_table_right], axis=1)
df_table.to_csv(f'Supp_Table03_low_beta.tsv', sep='\t', index=False)

Processing Basic model...
Processing Empty-room model...
Processing Cardiac model...
Processing 6-covariate model...
Processing Basic model...
Processing Empty-room model...
Processing Cardiac model...
Processing 6-covariate model...


In [ ]:
parameter = 'high_beta_band_power'
df_table_left = create_simple_table(df_source, parameter, 'High-Beta Power')
df_table_right = create_simple_table(df_source, 'high_beta_peak_freq', 'High-Beta Frequency')

df_table = pd.concat([df_table_left, df_table_right], axis=1)
df_table.to_csv(f'Supp_Table04_high_beta.tsv', sep='\t', index=False)

Processing Basic model...
Processing Empty-room model...
Processing Cardiac model...
Processing 6-covariate model...
Processing Basic model...
Processing Empty-room model...
Processing Cardiac model...
Processing 6-covariate model...


In [ ]:
def create_model_table(df_source, model_label):
    # Extracts the statistics for all the parameters and effects for a given model and creates a table as DataFrame. This is used for the control analyses (supplementary tables).

    # model_label is the label of the model in the source data file (e.g., 'basic_main', 'onecov_emptyroom', etc.)
    
    # Create a new pandas DataFrame to hold the table data (4 columns)
    column_names = ['Parameter / Age effect', f'Parameter Estimate, a.u./Hz p.a. (SD)', 'T-statistic (df)', 'P-value uncorrected (corrected)']   

    parameters = ['exponent']
    bands = ['theta', 'alpha', 'beta', 'low_alpha', 'high_alpha', 'low_beta', 'high_beta', 'gamma']

    for band in bands:
        for measure in ['band_power', 'peak_freq']:
            parameters.append(f'{band}_{measure}')

    parameters_new = [parameters[0]]

    for i in range(1, len(parameters), 2):
        parameters_new.extend([parameters[i+1], parameters[i]])

    parameters = parameters_new

    effects = ['A0', 'dA', 'A0:dA'] # as they will appear in the new table (paper)
    effects_labels = ['Age0', 'deltaAge', 'Age0:deltaAge'] # as they appear in the source data file 
    
    # Table data
    df_list = []
    for p, parameter_label in enumerate(parameters):
        print(f'Processing {parameter_label} model...')

        parameter = parameter_label.replace('_', ' ').title() # format the parameter label for the new table
        parameter = parameter.replace('Peak Freq', 'Peak Frequency') # format the parameter label for the new table

        df_tmp_list = [pd.DataFrame.from_dict({0: [parameter, '', '', '']}, orient='index', columns=column_names)]

        for e, effect in enumerate(effects):

            df_current = df_source.loc[(df_source['Parameter'] == parameter_label) & (df_source['Effect'] == effects_labels[e])]
                        
            tmp_dict = {
                e+1: [effect, f"{process_beta(df_current [f'{model_label}_Estimate'].values[0])} ({process_beta(df_current [f'{model_label}_SE'].values[0])})",
                        f"{round(df_current [f'{model_label}_T-stat'].values[0], 2)} ({round(df_current [f'{model_label}_DF'].values[0], 1)})",
                        f"{process_pval(df_current [f'{model_label}_P-val'].values[0])} ({process_pval(df_current [f'{model_label}_P-corrected'].values[0])})"]
            }

            tmp_df = pd.DataFrame.from_dict(tmp_dict, orient='index', columns=column_names)
            tmp_df.replace('nan (nan)', np.nan, inplace=True) # replace the 'nan (nan)' strings with actual NaN values so they can be dropped
            tmp_df.dropna(inplace=True) # drop rows with missing values (i.e., effects with no data)
            if tmp_df.shape[0] == 0:
                print(f'No data for {parameter_label} - {effects_labels[e]} in model {model_label}. Skipping...')
                continue
            df_tmp_list.append(tmp_df)

        if len(df_tmp_list) < 4: # if there is less than 3 effects with data (i.e., only the parameter name row), skip the parameter
            print(f'No data for any effect for {parameter_label} in model {model_label}. Skipping...')
            continue
        df_tmp = pd.concat(df_tmp_list)
        df_list.append(df_tmp)
                        
    df_table = pd.concat(df_list, ignore_index=True)

    return df_table

model_label = 'control_emptyroom'
df_table = create_model_table(df_source, model_label)
df_table.to_csv(f'Supp_Table05_{model_label}.tsv', sep='\t', index=False)

Processing exponent model...
Processing theta_peak_freq model...
Processing theta_band_power model...
Processing alpha_peak_freq model...
Processing alpha_band_power model...
Processing beta_peak_freq model...
Processing beta_band_power model...
Processing low_alpha_peak_freq model...
Processing low_alpha_band_power model...
Processing high_alpha_peak_freq model...
Processing high_alpha_band_power model...
Processing low_beta_peak_freq model...
Processing low_beta_band_power model...
Processing high_beta_peak_freq model...
Processing high_beta_band_power model...
Processing gamma_peak_freq model...
Processing gamma_band_power model...


In [ ]:
model_label = 'control_ECGchannel'
df_table = create_model_table(df_source, model_label)
df_table.to_csv(f'Supp_Table06_{model_label}.tsv', sep='\t', index=False)

Processing exponent model...
Processing theta_peak_freq model...
Processing theta_band_power model...
Processing alpha_peak_freq model...
No data for alpha_peak_freq - Age0 in model control_ECGchannel. Skipping...
No data for alpha_peak_freq - deltaAge in model control_ECGchannel. Skipping...
No data for alpha_peak_freq - Age0:deltaAge in model control_ECGchannel. Skipping...
No data for any effect for alpha_peak_freq in model control_ECGchannel. Skipping...
Processing alpha_band_power model...
No data for alpha_band_power - Age0 in model control_ECGchannel. Skipping...
No data for alpha_band_power - deltaAge in model control_ECGchannel. Skipping...
No data for alpha_band_power - Age0:deltaAge in model control_ECGchannel. Skipping...
No data for any effect for alpha_band_power in model control_ECGchannel. Skipping...
Processing beta_peak_freq model...
No data for beta_peak_freq - Age0 in model control_ECGchannel. Skipping...
No data for beta_peak_freq - deltaAge in model control_ECGcha

No data for beta_band_power - Age0 in model control_ECGchannel. Skipping...
No data for beta_band_power - deltaAge in model control_ECGchannel. Skipping...
No data for beta_band_power - Age0:deltaAge in model control_ECGchannel. Skipping...
No data for any effect for beta_band_power in model control_ECGchannel. Skipping...
Processing low_alpha_peak_freq model...
No data for low_alpha_peak_freq - Age0 in model control_ECGchannel. Skipping...
No data for low_alpha_peak_freq - deltaAge in model control_ECGchannel. Skipping...
No data for low_alpha_peak_freq - Age0:deltaAge in model control_ECGchannel. Skipping...
No data for any effect for low_alpha_peak_freq in model control_ECGchannel. Skipping...
Processing low_alpha_band_power model...
No data for low_alpha_band_power - Age0 in model control_ECGchannel. Skipping...
No data for low_alpha_band_power - deltaAge in model control_ECGchannel. Skipping...
No data for low_alpha_band_power - Age0:deltaAge in model control_ECGchannel. Skipping.

/tmp/ipykernel_843481/1884679148.py:47: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tmp_df.replace('nan (nan)', np.nan, inplace=True) # replace the 'nan (nan)' strings with actual NaN values so they can be dropped


In [ ]:
model_label = 'control_cardiacMEGonly'
df_table = create_model_table(df_source, model_label)
df_table.to_csv(f'Supp_Table07_{model_label}.tsv', sep='\t', index=False)

Processing exponent model...
Processing theta_peak_freq model...
Processing theta_band_power model...
Processing alpha_peak_freq model...
Processing alpha_band_power model...
Processing beta_peak_freq model...
Processing beta_band_power model...
Processing low_alpha_peak_freq model...
No data for low_alpha_peak_freq - Age0 in model control_cardiacMEGonly. Skipping...
No data for low_alpha_peak_freq - deltaAge in model control_cardiacMEGonly. Skipping...
No data for low_alpha_peak_freq - Age0:deltaAge in model control_cardiacMEGonly. Skipping...
No data for any effect for low_alpha_peak_freq in model control_cardiacMEGonly. Skipping...
Processing low_alpha_band_power model...
No data for low_alpha_band_power - Age0 in model control_cardiacMEGonly. Skipping...
No data for low_alpha_band_power - deltaAge in model control_cardiacMEGonly. Skipping...
No data for low_alpha_band_power - Age0:deltaAge in model control_cardiacMEGonly. Skipping...
No data for any effect for low_alpha_band_power 

/tmp/ipykernel_843481/1884679148.py:47: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tmp_df.replace('nan (nan)', np.nan, inplace=True) # replace the 'nan (nan)' strings with actual NaN values so they can be dropped
/tmp/ipykernel_843481/1884679148.py:47: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tmp_df.replace('nan (nan)', np.nan, inplace=True) # replace the 'nan (nan)' strings with actual NaN values so they can be dropped


In [ ]:
model_label = 'control_ignorepeaks'
df_table = create_model_table(df_source, model_label)
df_table.to_csv(f'Supp_Table09_{model_label}.tsv', sep='\t', index=False)

Processing exponent model...
Processing theta_peak_freq model...
Processing theta_band_power model...
Processing alpha_peak_freq model...
Processing alpha_band_power model...
Processing beta_peak_freq model...
Processing beta_band_power model...
Processing low_alpha_peak_freq model...
Processing low_alpha_band_power model...
Processing high_alpha_peak_freq model...
Processing high_alpha_band_power model...
Processing low_beta_peak_freq model...
Processing low_beta_band_power model...
Processing high_beta_peak_freq model...
No data for high_beta_peak_freq - Age0 in model control_ignorepeaks. Skipping...
No data for high_beta_peak_freq - deltaAge in model control_ignorepeaks. Skipping...
No data for high_beta_peak_freq - Age0:deltaAge in model control_ignorepeaks. Skipping...
No data for any effect for high_beta_peak_freq in model control_ignorepeaks. Skipping...
Processing high_beta_band_power model...
No data for high_beta_band_power - Age0 in model control_ignorepeaks. Skipping...
No 

/tmp/ipykernel_843481/1884679148.py:47: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  tmp_df.replace('nan (nan)', np.nan, inplace=True) # replace the 'nan (nan)' strings with actual NaN values so they can be dropped


In [ ]:
model_label = 'control_nointerpol'
df_table = create_model_table(df_source, model_label)
df_table.to_csv(f'Supp_Table10_{model_label}.tsv', sep='\t', index=False)

Processing exponent model...
Processing theta_peak_freq model...
Processing theta_band_power model...
Processing alpha_peak_freq model...
Processing alpha_band_power model...
Processing beta_peak_freq model...
Processing beta_band_power model...
Processing low_alpha_peak_freq model...
Processing low_alpha_band_power model...
Processing high_alpha_peak_freq model...
Processing high_alpha_band_power model...
Processing low_beta_peak_freq model...
Processing low_beta_band_power model...
Processing high_beta_peak_freq model...
Processing high_beta_band_power model...
Processing gamma_peak_freq model...
Processing gamma_band_power model...


In [ ]:
df_source = pd.read_csv(os.path.join(datadir, f'tables_source_data_mag.tsv'), sep='\t')
model_label = 'basic_main'
df_table = create_model_table(df_source, model_label)
df_table.to_csv(f'Supp_Table11_{model_label}_mag.tsv', sep='\t', index=False)

Processing exponent model...
Processing theta_peak_freq model...
Processing theta_band_power model...
Processing alpha_peak_freq model...
Processing alpha_band_power model...
Processing beta_peak_freq model...
Processing beta_band_power model...
Processing low_alpha_peak_freq model...
Processing low_alpha_band_power model...
Processing high_alpha_peak_freq model...
Processing high_alpha_band_power model...
Processing low_beta_peak_freq model...
Processing low_beta_band_power model...
Processing high_beta_peak_freq model...
Processing high_beta_band_power model...
Processing gamma_peak_freq model...
Processing gamma_band_power model...


### Tables LME covariates

In [ ]:
def create_table_covariates():
    cov_list = ['headposx', 'headposy', 'headposz', 'headmov']
    cov_labels = ['Head position (x)', 'Head position (y)', 'Head position (z)', 'Head movement']

    # --- Create a new DataFrame to hold the table data (4 columns) ---
    column_names = ['Covariate / Age effect', 'Parameter Estimate, mm p.a. (SD)', 'T-statistic (df)', 'P-value uncorrected (corrected*)']

    df_table = pd.DataFrame(columns=column_names)

    # Loop through variables and add rows to data tuple
    for idx, cov in enumerate(cov_list):

        cov_label = cov_labels[idx]

        resultsfile = f'{cov}_lme_results.tsv'
        resfilepath = os.path.join(datadir, resultsfile)
        results_df = pd.read_csv(resfilepath, sep='\t').set_index('Unnamed: 0')

        df_tmp_list = [pd.DataFrame.from_dict({0: [cov_label, '', '', '']}, orient='index', columns=column_names)]

        effects = ['A0', 'dA', 'A0:dA'] # as they will appear in the new table (paper)

        for e, effect in enumerate(effects):
            tmp_dict = {
                e+1: [effect, f'{process_beta(results_df.at[effect,'Estimate'])} ({process_beta(results_df.at[effect,'SE'])})', f'{round(results_df.at[effect,'T-stat'], 2)} ({round(results_df.at[effect,'DF'], 1)})', f'{process_pval(results_df.at[effect,'P-val'])} ({process_pval(results_df.at[effect,'P-val']*len(cov_list))})'],
            }
            df_tmp = pd.DataFrame.from_dict(tmp_dict, orient='index', columns=column_names)
            df_tmp_list.append(df_tmp)    

        df_table = pd.concat([df_table] + df_tmp_list, ignore_index=True)          

    return df_table


df_table = create_table_covariates()
df_table.to_csv(f'Supp_Table08_covariates.tsv', sep='\t', index=False)